<a href="https://colab.research.google.com/github/napoles-uach/knowledgeGraphLLMHackathon/blob/main/hackathon_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U pyvis jinja2 pandas openpyxl networkx fuzzywuzzy python-Levenshtein

To run this code, you need to have the file "LLM Hackathon 2025 Themes.xlsx" 👉 [file](https://docs.google.com/spreadsheets/d/1JW3GjmXceA409NR363RrasZ4_Ml9uoGb2n5URY-pVv4/edit?usp=sharing)

In [2]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import re
from fuzzywuzzy import fuzz

EXCEL_PATH = "LLM Hackathon 2025 Themes.xlsx"
SHEET = "Main"

# === 1️⃣ Detectar fila con encabezados ===
df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET, header=None)

header_row_idx = None
for i in range(min(30, len(df_raw))):
    if df_raw.iloc[i].astype(str).str.strip().eq("Project/Team Name").any():
        header_row_idx = i
        break

if header_row_idx is None:
    raise ValueError("❌ No se encontró la fila con encabezados (buscando 'Project/Team Name').")

df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET, header=header_row_idx)
df.columns = df.columns.map(str.strip)

# === 2️⃣ Detectar nombres de columnas relevantes ===
def find_col(possibles):
    for c in df.columns:
        for p in possibles:
            if p.lower() in c.lower():
                return c
    return None

col_project = find_col(["Project/Team Name", "Project", "Team"])
col_primary = find_col(["Primary Theme", "PRIMARY THEME"])
col_secondary = find_col(["Secondary Themes", "Secondary Theme"])

rename_map = {}
if col_project: rename_map[col_project] = "Project/Team Name"
if col_primary: rename_map[col_primary] = "Primary Theme"
if col_secondary: rename_map[col_secondary] = "Secondary Themes"

df = df.rename(columns=rename_map)
df = df[["Project/Team Name", "Primary Theme", "Secondary Themes"]].copy()

# === 3️⃣ Función robusta para dividir solo temas secundarios ===
def split_secondary(val):
    """Divide las celdas de temas secundarios, respetando comillas y comas internas."""
    if pd.isna(val):
        return []

    text = str(val).strip()
    if not text:
        return []

    text = text.strip("[]").replace("“", '"').replace("”", '"')

    # Detectar fragmentos entre comillas
    quoted_items = re.findall(r'"([^"]+)"', text)
    if not quoted_items:
        quoted_items = re.findall(r"'([^']+)'", text)

    if quoted_items:
        remainder = re.sub(r'"[^"]+"', "", text)
        remainder = re.sub(r"'[^']+'", "", remainder)
        remainder_parts = [p.strip() for p in re.split(r"[,;|]", remainder) if p.strip()]
        items = quoted_items + remainder_parts
    else:
        items = [p.strip() for p in re.split(r"[,;|]", text) if p.strip()]

    return list(dict.fromkeys(items))  # eliminar duplicados manteniendo orden

# === 4️⃣ Crear grafo ===
G = nx.Graph()

TYPE_STYLE = {
    "project": {"color": "#3498db", "shape": "dot", "size": 15},     # Azul
    "theme": {"color": "#2ecc71", "shape": "dot", "size": 28},       # Verde (Primario)
    "secondary_theme": {"color": "#9b59b6", "shape": "dot", "size": 20}  # Morado (Secundario)
}

def add_node_safe(g, node_id, label, ntype):
    """Agrega un nodo único (por id), pero con etiqueta limpia para mostrar."""
    if not node_id:
        return

    style = TYPE_STYLE[ntype]
    if not g.has_node(node_id):
        g.add_node(
            node_id,
            label=label,  # lo que se muestra
            type=ntype,
            color=style["color"],
            shape=style["shape"],
            size=style["size"]
        )

# === 5️⃣ Construcción del grafo ===
for _, row in df.iterrows():
    project = str(row.get("Project/Team Name", "")).strip()
    if not project:
        continue

    add_node_safe(G, project, project, "project")

    # --- Tema primario (uno por proyecto)
    primary = str(row.get("Primary Theme", "")).strip()
    if primary:
        primary_id = f"{primary}_P"  # id interno único
        add_node_safe(G, primary_id, primary, "theme")
        G.add_edge(project, primary_id, relation="primary_theme")

    # --- Temas secundarios (múltiples)
    secondary = row.get("Secondary Themes")
    if pd.notna(secondary):
        for t in split_secondary(secondary):
            secondary_id = f"{t}_S"
            add_node_safe(G, secondary_id, t, "secondary_theme")
            G.add_edge(project, secondary_id, relation="secondary_theme")

print(f"\n✅ Nodos totales: {G.number_of_nodes()} | Aristas: {G.number_of_edges()}")

# === 6️⃣ Exportar relaciones ===
edges = []
for src, dst, data in G.edges(data=True):
    edges.append([src, dst, data["relation"]])
edges_df = pd.DataFrame(edges, columns=["Source", "Target", "Relation"])
edges_df.to_csv("hackathon_edges.csv", index=False)
print("📄 Relaciones exportadas a 'hackathon_edges.csv'")

# === 7️⃣ Visualización ===
net = Network(height="750px", width="100%", bgcolor="#111111", font_color="white")
net.from_nx(G)
net.force_atlas_2based(gravity=-50)

try:
    net.show("hackathon_graph.html")
except Exception:
    net.save_graph("hackathon_graph.html")

print("✅ Grafo generado: hackathon_graph.html")



✅ Nodos totales: 133 | Aristas: 158
📄 Relaciones exportadas a 'hackathon_edges.csv'
hackathon_graph.html
✅ Grafo generado: hackathon_graph.html
